In [ ]:
import json
import requests
from tqdm import tqdm
from nltk.stem.snowball import SnowballStemmer

stemmer = SnowballStemmer("russian")

API_URL = "http://localhost:11434/api/chat"
API_URL_GENERATE = "http://localhost:11434/api/generate"
MODEL_NAME = "qwen2.5:14b"



def call_llm(system_prompt, user_prompt):
    payload = {
        "model": MODEL_NAME,
        "prompt": user_prompt, 
        "system": system_prompt, 
        "stream": False,
        "keep_alive": 0,
        "options": {
            "temperature": 0.0,
            "seed": 42,
            "num_ctx": 6144,
            "repetition_penalty": 1.05,
            "num_predict": 2048, 
        },
    }
    r = requests.post(API_URL_GENERATE, json=payload, timeout=120)
    return r.json()["response"]


def extract_terms_llm(text):
    system_prompt = """
Ты — инструмент для извлечения медицинских терминов. 
Твоя задача: найти в тексте и выписать все медицинские сущности.

ПРАВИЛА:
1. Выписывай термины как они есть в тексте. Просто КОПИРУЙ(подстрока)
2. НЕ сокращай и НЕ расшифровывай: если в тексте "ОМЛ" — пиши "ОМЛ", если "острый миелоидный лейкоз" — пиши "острый миелоидный лейкоз".
3. Извлекай ВСЁ: болезни, симптомы, лекарства, сокращения (ЩФ, ГГТП, ИСТ), генетические маркеры (FLT3, 46XX), процедуры, протоколы(FluBu10, AML-MRD2018) и нормальные/патологические показатели.
4. Если термин содержит степень или локализацию — извлекай вместе с ними (н-р: ОМЛ (м5а вариант), "РТПХ кожи", "мукозит 2 степени").
5. Извлекай анатомические зоны (печень, илеоцекальный угол, правая височная доля, левое легкое)
6. Используй только РУССКИЙ язык (за исключением латинских названий лекарств или генов).
7. НЕ извлекай даты, дозировки.

ФОРМАТ(НИЧЕГО кроме этого не пиши):
["термин1", "термин2", ...]"""

    

    #user_prompt = f"""{text.replace('\\', '/')}"""
    user_prompt = text.replace('\\', '/')
    content = call_llm(system_prompt, user_prompt)
    #print(content)

    try:
        content_fixed = content.replace("\\", "/")
        content_fixed = content_fixed.replace("Ответ: ", "")
        data = json.loads(content_fixed)
        # for item in data:
        return data
        # return [item["term"] for item in data if "term" in item]
    except:
        # fallback
        return []

In [19]:
import pandas as pd
from splitter import chunk_text_by_sentences

df_full = pd.read_parquet("../data/data2teach.parquet")


texts = df_full['descr'].to_list()
for i in range(len(texts)):
    try:
        texts[i] = texts[i].split('\n')[1]
    except IndexError:
        pass


In [ ]:
texts[0]

In [20]:
extract_terms_llm(texts[0])

['отек левой ноги',
 'компрессией повздошных сосудов слева',
 'альвеолярная РМС',
 'ПХТ по CWS-2009',
 'РТПХ',
 'луч',
 'радикальная операция',
 'РМСА 3 степень патоморфоза',
 'СОД41,4Гр',
 'СОД50,4Гр',
 'винорельбин',
 'эндоксан',
 'II системный рецидив',
 'опухолевый лимфостаз',
 'метастазы в кожу',
 'метастазы в лимфоузлы',
 'увеличение левого яичка',
 'паллиативная ПХТ',
 'Avastin',
 'Temodal',
 'Irinotekan',
 'VCR',
 'полный ответ',
 'химиочувствительный рецидив',
 'ВДХТ',
 'ауто-ТГСК',
 'цитопения',
 'миелоэксфузия']

In [25]:
import nltk
from nltk.tokenize import sent_tokenize


data = []

for text in tqdm(texts):
    
    for sent in sent_tokenize(text, language='russian'):
        d = dict()
        d["chunk"] = sent
        d["terms"] = extract_terms_llm(sent)

        data.append(d)

import pickle

with open("data2teach.pkl", "wb") as file:
    pickle.dump(data, file)

100%|██████████| 100/100 [1:00:50<00:00, 36.51s/it]


In [4]:
import pandas as pd

df = pd.read_parquet("terms_with_val300.parquet")
df

,chunk,terms,truth
0,", Пациентка 28 лет с диагнозом: Острый миелоид...","[острый миелоидный лейкоз, аллогенная ткск, му...","[острый миелоидный лейкоз, М2 FAB, 47XX+22, FL..."
1,Значимые изменения ST-T не зарегистрированы пр...,"[ST-T изменения, QT интервал, корригированнный...","[ST-T изменения, QT интервал, корригированнный..."
2,Подмышечные лимфатические узлы размерами до 11...,"[лимфатические узлы, онколог, экстрамедулярный...","[лимфатические узлы, онколог, экстрамедулярный..."
3,", Пациентка, 2лет. c диагнозом:Острый миелобла...","[острый миелобластный лейкоз, м5а вариант, пер...","[острый миелобластный лейкоз, м5а вариант, пер..."
4,", Пациентка 1 года, с диагнозом врожденный ОЛЛ...","[ОЛЛ, ТКМ, химиотерапия, рецидив, ретробульбар...","[врожденный ОЛЛ, гаплоидентичная ТКМ, комбинир..."
...,...,...,...
295,С 16.09.16 начата терапия ГКС 1 мг/кг под защи...,"[ГКС, РТПХ, билирубин, такролимус, вифенд, КТ ...",None
296,", Пациент 61 года с диагнозом: Симптоматическа...","[симптоматическая множественная миелома, плазм...",None
297,", ЭТАПНЫЙ ЭПИКРИЗ, Пациент 20 лет с диагнозом ...","[ОЛЛ, стернальная пункции, люмбальная пункции,...",None
298,", Пациент 3 мес, с диагнозом: Острый лимфоблас...","[острый лимфобластный лейкоз, инфузионная тера...",None


In [32]:
sentences = []
for d in data:
    sentences.append(d['chunk'])
sentences = set(sentences)

1318

In [33]:
for i, row in tqdm(df.iterrows()):
    if i < 99:
        continue
    for sent in sent_tokenize(row['chunk'], language='russian'):
        if sent in sentences:
            continue
        d = dict()
        d["chunk"] = sent
        d["terms"] = extract_terms_llm(sent)

        sentences.add(sent)
        data.append(d)

300it [55:40, 11.13s/it]


In [34]:
with open("data2teach.pkl", "wb") as file:
    pickle.dump(data, file)

In [3]:
import pickle 

with open("data2teach.pkl", "rb") as file:
    data = pickle.load(file)

In [ ]:
data

In [38]:
seen_chunks = set()
unique_data = []

for item in data:
    chunk_value = item["chunk"]
    if chunk_value not in seen_chunks:
        seen_chunks.add(chunk_value)
        unique_data.append(item)


In [ ]:
unique_data

In [ ]:
import re
from difflib import SequenceMatcher
from nltk.stem.snowball import SnowballStemmer

stemmer = SnowballStemmer("russian")

def ultra_normalize(text):
    """Приводит текст к единому виду: латиница, без ё, без мусора"""
    if not text: return ""
  
    cyr = "АВЕКМНОРСТУХаекорсху"
    lat = "ABEKMHOPCTYXaekopcxy"
    trans = str.maketrans(cyr, lat)
    
    t = text.lower().replace('ё', 'е').translate(trans)
   
    t = t.replace('deletion', 'делеция').replace('mts', 'мтс')
    return t

def get_stems(text):
    """Получает список очищенных стемов"""
    words = re.findall(r'\b\w+\b', ultra_normalize(text))
    return [stemmer.stem(w) for w in words]

def find_best_span(text: str, term: str, max_gap=5):
    """
    Ищет в тексте окно, где максимально кучно лежат стемы из термина.
    max_gap: сколько 'левых' слов может быть между словами термина.
    """
    
    norm_text = ultra_normalize(text)
    norm_term = ultra_normalize(term)
    
    idx = norm_text.find(norm_term)
    if idx != -1:
        return idx, idx + len(term), text[idx:idx+len(term)]

   
    term_stems = get_stems(term)
    if not term_stems: return None, None, None
    

    words_in_text = []
    for m in re.finditer(r'\w+', text):
        words_in_text.append({
            'start': m.start(),
            'end': m.end(),
            'stem': stemmer.stem(ultra_normalize(m.group()))
        })

    best_match = None
    max_found_stems = 0
    min_window_len = float('inf')


    for i in range(len(words_in_text)):
        found_stems = set()
        current_window_words = []
        
     
        for j in range(i, min(i + len(term_stems) + max_gap, len(words_in_text))):
            word_stem = words_in_text[j]['stem']
            
          
            for ts in term_stems:
                if word_stem == ts or (len(ts) > 3 and (ts in word_stem or word_stem in ts)):
                    found_stems.add(ts)
                    current_window_words.append(words_in_text[j])
                    break
            
   
            if len(found_stems) >= max(1, len(term_stems) * 0.7):
                current_score = len(found_stems)
                window_start = current_window_words[0]['start']
                window_end = current_window_words[-1]['end']
                window_len = window_end - window_start
            
                if (current_score > max_found_stems) or \
                   (current_score == max_found_stems and window_len < min_window_len):
                    max_found_stems = current_score
                    min_window_len = window_len
                    best_match = (window_start, window_end, text[window_start:window_end])

    return best_match if best_match else (None, None, None)

def normalize_terms_in_data(data, verbose=True):
    """
    Принимает список словарей с ключами 'chunk' и 'terms'.
    Возвращает новый список, где каждый термин заменён на реальную подстроку из chunk.
    Термины, которые не удалось найти, удаляются.
    """
    new_data = []
    total_terms = 0
    found_terms = 0

    for idx, item in enumerate(data):
        chunk = item['chunk']
        original_terms = item['terms']
        corrected_terms = []

        for term in original_terms:
            total_terms += 1
            start, end, matched = find_best_span(chunk, term)
            if matched:
                corrected_terms.append(matched)
                found_terms += 1
            else:
                if verbose:
                    snippet = chunk[:70] + "..." if len(chunk) > 70 else chunk
                    print(f"[{idx}] Термин не найден: '{term}' -> в тексте: {chunk}\n")


        new_data.append({
            'chunk': chunk,
            'terms': corrected_terms
        })

    if verbose:
        print(f"\nСтатистика: найдено {found_terms} из {total_terms} терминов ({100*found_terms/total_terms:.1f}%)")

    return new_data



# Нормализуем
terms_new = normalize_terms_in_data(unique_data, verbose=True)

   

In [59]:
terms_df = pd.DataFrame(terms_new)
terms_df.to_excel("terms2teach.xlsx", index=False)

In [ ]:
import ast
terms_df = pd.read_excel("terms2teach.xlsx")
terms_df = terms_df.dropna()

terms_new = terms_df.to_dict(orient="records")

for i in range(len(terms_new)):
    terms_new[i]['terms'] = ast.literal_eval(terms_new[i]['terms'])

for i in range(len(terms_new)):
    temp = terms_new[i].get('terms')
    r = []
    #print(temp)
    for j in range(len(temp)):
        if len(temp[j].strip()) == 1:
            continue
        r.append(temp[j])
    terms_new[i]["terms"] = r
terms_new

In [10]:
(terms_new[1].get('terms'))

['биопсия лимфоузла', 'гистология', 'ИГХ', 'альвеолярная РМС']

In [ ]:


import json
import re
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModelForTokenClassification, TrainingArguments, Trainer
from torch.utils.data import Dataset
from difflib import SequenceMatcher
import evaluate


seqeval = evaluate.load("seqeval")

BMT_TERMS = {
    'трансплантация костного мозга', 'тгск', 'аллогенная тгск', 'аутологичная тгск',
    'резистентное течение', 'прогрессия основного заболевания', 'индукционная химиотерапия',
    'высокодозная пхт', 'циторедукция', 'синдром дауна', 'экстрамедуллярное поражение',
    'инфильтрация зрительных нервов', 'мукозит', 'оРТПХ', 'хроническая ртпх',
    'фебрильная нейтропения', 'приживление трансплантата', 'инфузия донорских лимфоцитов',
    'такролимус', 'метотрексат', 'циклоспорин', 'бусульфан', 'флударабин', 'аллогенная трасплантация гомопоэтических стволовых клеток',
    'трансплантат', 'кондиционирование', 'ремиссия', 'рецидив', 'аллогенный', 'аллоТГСК'
}


def normalize_for_search(s: str) -> str:
    """Приводим к нижнему регистру, убираем пунктуацию по краям и лишние пробелы"""
    s = s.strip().lower()
    s = re.sub(r'^[^\w\s]+|[^\w\с]+$', '', s)  
    s = re.sub(r'\s+', ' ', s)             
    return s

def is_valid_term(term: str, min_words: int = 1, max_words: int = 15) -> bool:
    """Фильтрует слишком короткие/длинные или бессмысленные термины"""
    words = term.split()
    if len(words) < min_words or len(words) > max_words:
        return False

    if len(words) == 1 and words[0] in {'и', 'в', 'на', 'с', 'к', 'у', 'от', 'до', 'по'}:
        return False
    return True

def split_long_term(term: str, max_words: int = 12) -> list:
    """
    Разбивает слишком длинный термин (например, целое предложение) на более короткие
    Пытается сохранить медицинскую значимость, обрезая по союзам/предлогам.
    """
    words = term.split()
    if len(words) <= max_words:
        return [term]
    
    candidates = []
    for i, w in enumerate(words):
        if w.lower() in {'и', 'а', 'но', 'с', 'без', 'для', 'до', 'из', 'к', 'на', 'по', 'при', 'про', 'через'}:
            candidates.append(i)
    if not candidates:
        return [' '.join(words[:max_words])]
    
    cut_pos = candidates[0]
    if cut_pos < 3:

        return [' '.join(words[:max_words])]
    return [' '.join(words[:cut_pos])]

def expand_terms_with_dictionary(text: str, original_terms: list) -> list:
    """Добавляет термины из словаря BMT_TERMS, если они присутствуют в тексте и не перекрываются"""
    text_lower = text.lower()
    expanded = set(original_terms)
    for dict_term in BMT_TERMS:
        if dict_term in text_lower:
            already_covered = False
            for ot in original_terms:
                if dict_term in ot.lower() or ot.lower() in dict_term:
                    already_covered = True
                    break
            if not already_covered:
                expanded.add(dict_term)
    return list(expanded)

def fuzzy_find(text_lower: str, norm_term: str, threshold: float = 0.85) -> tuple:
    """
    Ищет приблизительное вхождение термина в тексте, используя SequenceMatcher.
    Возвращает (start, end) или (-1, -1), если не найдено.
    """
    idx = text_lower.find(norm_term)
    if idx != -1:
        return (idx, idx + len(norm_term))
    
    term_len = len(norm_term)
    best_ratio = 0
    best_start = -1
    for start in range(max(0, len(text_lower) - term_len - 5)):
        end = min(len(text_lower), start + term_len + 5)
        candidate = text_lower[start:end]
        ratio = SequenceMatcher(None, norm_term, candidate).ratio()
        if ratio > best_ratio and ratio >= threshold:
            best_ratio = ratio
            best_start = start
            best_end = end
    if best_start != -1:
        return (best_start, best_end)
    return (-1, -1)


import nltk
from nltk.stem.snowball import SnowballStemmer
stemmer = SnowballStemmer("russian")

def normalize_and_stem(phrase: str) -> str:
    """Возвращает строку из стемов слов, разделённых пробелами"""
    words = re.findall(r'\b\w+\b', phrase.lower())
    stemmed = [stemmer.stem(w) for w in words]
    return ' '.join(stemmed)

def convert_terms_to_bio(text, terms, tokenizer, max_length=512):
    clean_terms = sorted(list(set(terms)), key=len, reverse=True)
    
    encoding = tokenizer(
        text,
        truncation=True,
        max_length=max_length,
        padding='max_length',
        return_offsets_mapping=True
    )
    
    offset_mapping = encoding['offset_mapping']

    labels = ['O'] * len(encoding['input_ids'])
    
    for term in clean_terms:
        if not term.strip(): continue
        
        start = 0
        while True:
            start = text.find(term, start)
            if start == -1:
                break
            end = start + len(term)
            
            is_left_clean = (start == 0) or not text[start-1].isalnum()
            is_right_clean = (end == len(text)) or not text[end].isalnum()

            if is_left_clean and is_right_clean:
                first_token_of_entity = True
                for i, (tok_start, tok_end) in enumerate(offset_mapping):
                 
                    if tok_start == tok_end: 
                        continue
                    
     
                    if tok_start < end and tok_end > start:
                        if first_token_of_entity:
                           
                            if labels[i] == 'O':
                                labels[i] = 'B-TERM'
                                first_token_of_entity = False
                        else:
                            if labels[i] == 'O':
                                labels[i] = 'I-TERM'
            
            start += 1 
    
    label2id = {'O': 0, 'B-TERM': 1, 'I-TERM': 2}
    return {
        'input_ids': encoding['input_ids'],
        'attention_mask': encoding['attention_mask'],
        'labels': [label2id[l] for l in labels]
    }


class NERDataset(Dataset):
    def __init__(self, features):
        self.features = features
    
    def __len__(self):
        return len(self.features)
    
    def __getitem__(self, idx):
        return {
            'input_ids': torch.tensor(self.features[idx]['input_ids'], dtype=torch.long),
            'attention_mask': torch.tensor(self.features[idx]['attention_mask'], dtype=torch.long),
            'labels': torch.tensor(self.features[idx]['labels'], dtype=torch.long)
        }

def prepare_dataset(data_list, tokenizer, max_length=512):
    """
    data_list: список dict с ключами 'chunk' (текст) и 'terms' (список терминов)
    """
    features = []
    for item in data_list:
        feat = convert_terms_to_bio(item['chunk'], item['terms'], tokenizer, max_length)
        features.append(feat)
    return NERDataset(features)


def train_ner(train_data, eval_data, model_name, output_dir="./binary_ner_model"):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    
   
    print("Проверка качества терминов в train_data...")
    total_terms = 0
    total_cleaned = 0
    for item in train_data:
        original = len(item['terms'])
        total_terms += original
       
        cleaned = [t for t in item['terms'] if is_valid_term(normalize_for_search(t), max_words=15)]
        total_cleaned += len(cleaned)
    print(f"Фильтрация: {total_terms} -> {total_cleaned} терминов (удалено {total_terms - total_cleaned})")
    
    train_dataset = prepare_dataset(train_data, tokenizer)
    eval_dataset = prepare_dataset(eval_data, tokenizer)
    
    label_list = ['O', 'B-TERM', 'I-TERM']
    num_labels = len(label_list)
    
    model = AutoModelForTokenClassification.from_pretrained(
        model_name,
        num_labels=num_labels,
        id2label={i: l for i, l in enumerate(label_list)},
        label2id={l: i for i, l in enumerate(label_list)}
    )
    
    training_args = TrainingArguments(
        output_dir=output_dir,
        learning_rate=2e-5,       
        per_device_train_batch_size=8, 
        per_device_eval_batch_size=8,
        num_train_epochs=4,          
        weight_decay=0.01,
        warmup_ratio=0.1,            
        eval_strategy="epoch",
        save_strategy="epoch",
        logging_dir="./logs",
        logging_steps=50,
        save_total_limit=2,
        fp16=True,              
        load_best_model_at_end=True,  
        metric_for_best_model="f1"  
    )
    
    def compute_metrics(p):
        predictions, labels = p
        predictions = np.argmax(predictions, axis=2)

     
        true_predictions = [
            [label_list[p] for (p, l) in zip(prediction, label) if l != -100]
            for prediction, label in zip(predictions, labels)
        ]
        true_labels = [
            [label_list[l] for (p, l) in zip(prediction, label) if l != -100]
            for prediction, label in zip(predictions, labels)
        ]

        results = seqeval.compute(predictions=true_predictions, references=true_labels)
        return {
            "precision": results["overall_precision"],
            "recall": results["overall_recall"],
            "f1": results["overall_f1"],
            "accuracy": results["overall_accuracy"],
        }
    
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        compute_metrics=compute_metrics,
    )
    
    trainer.train()
    trainer.save_model()
    tokenizer.save_pretrained(output_dir)
    print(f"Модель сохранена в {output_dir}")
    return trainer


def extract_terms(text, model_path, aggregation="simple"):
    from transformers import pipeline
    nlp = pipeline("ner", model=model_path, tokenizer=model_path, aggregation_strategy=aggregation)
    results = nlp(text)
    terms = [res['word'] for res in results if res['entity_group'] == 'TERM']
    # убираем дубликаты, сохраняя порядок
    seen = set()
    unique_terms = []
    for t in terms:
        if t not in seen:
            seen.add(t)
            unique_terms.append(t)
    return unique_terms



In [ ]:
from sklearn.model_selection import train_test_split

# terms_new — это результат работы вашего normalize_terms_in_data
train_data, eval_data = train_test_split(terms_new, test_size=0.15, random_state=42)

# И теперь запускаем обучение
trainer = train_ner(train_data, eval_data, model_name="alexyalunin/RuBioRoBERTa", output_dir="binary_ner_model_last")